In [1]:
# ==========================================================
# HEARTGUARD - DATA LOADING & INITIAL VALIDATION MODULE
# MULTI DATASET INDEPENDENT MODEL TRAINING
# ==========================================================

import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# ----------------------------------------------------------
# PATHS
# ----------------------------------------------------------

BASE_PATH = os.path.join("..", "data", "processed")
MODEL_PATH = os.path.join("..", "backend", "models")
os.makedirs(MODEL_PATH, exist_ok=True)

# ----------------------------------------------------------
# GENERIC TRAINING FUNCTION
# ----------------------------------------------------------

def train_model(df, target_column, model_name):

    print(f"\n🚀 Training {model_name} model")

    X = df.drop(columns=[target_column])
    y = df[target_column]

    # Imputation
    imputer = SimpleImputer(strategy="median")
    X = imputer.fit_transform(X)

    # Scaling
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Model
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    print(f"Accuracy: {acc:.4f}")
    print(f"ROC-AUC : {auc:.4f}")

    # Save artifacts
    joblib.dump(model, os.path.join(MODEL_PATH, f"{model_name}_model.pkl"))
    joblib.dump(scaler, os.path.join(MODEL_PATH, f"{model_name}_scaler.pkl"))
    joblib.dump(imputer, os.path.join(MODEL_PATH, f"{model_name}_imputer.pkl"))

    print(f"✅ {model_name} model saved")

    return auc


# ----------------------------------------------------------
# 1️⃣ FRAMINGHAM MODEL
# ----------------------------------------------------------

framingham = pd.read_csv(os.path.join(BASE_PATH, "framingham_clean_v1.csv"))

framingham_df = framingham[
    ["age","male","bmi","totchol","sysbp","diabp",
     "glucose","currentsmoker","prevalenthyp","tenyearchd"]
].dropna()

auc_f = train_model(framingham_df, "tenyearchd", "framingham")


# ----------------------------------------------------------
# 2️⃣ CARDIOVASCULAR MODEL
# ----------------------------------------------------------

cardio = pd.read_csv(os.path.join(BASE_PATH, "cardiovascular_clean_v1.csv"))

# Convert age from days to years
cardio["age"] = cardio["age"] / 365

# Calculate BMI
cardio["bmi"] = cardio["weight"] / ((cardio["height"]/100) ** 2)

cardio_df = cardio[
    ["age","gender","bmi","ap_hi","ap_lo",
     "cholesterol","smoke","cardio"]
]

auc_c = train_model(cardio_df, "cardio", "cardio")


# ----------------------------------------------------------
# 3️⃣ HEART ATTACK MODEL
# ----------------------------------------------------------

heart_attack = pd.read_csv(os.path.join(BASE_PATH, "heart_attack_clean_v1.csv"))

# Split blood pressure
bp_split = heart_attack["blood_pressure"].str.split("/", expand=True)
heart_attack["sysbp"] = bp_split[0].astype(float)
heart_attack["diabp"] = bp_split[1].astype(float)

heart_attack_df = heart_attack[
    ["age","cholesterol","sysbp","diabp",
     "bmi","smoking","heart_attack_risk"]
]

auc_ha = train_model(heart_attack_df, "heart_attack_risk", "heart_attack")


# ----------------------------------------------------------
# 4️⃣ HEART FAILURE MODEL
# ----------------------------------------------------------

heart_failure = pd.read_csv(os.path.join(BASE_PATH, "heart_failure_clean_v1.csv"))

heart_failure_df = heart_failure[
    ["age","ejection_fraction","serum_creatinine",
     "diabetes","high_blood_pressure","death_event"]
]

auc_hf = train_model(heart_failure_df, "death_event", "heart_failure")


# ----------------------------------------------------------
# 5️⃣ DIABETES MODEL
# ----------------------------------------------------------

diabetes = pd.read_csv(os.path.join(BASE_PATH, "diabetes_binary_clean_v1.csv"))

diabetes_df = diabetes[
    ["age","bmi","highbp","highchol",
     "smoker","physactivity","heartdiseaseorattack"]
]

auc_d = train_model(diabetes_df, "heartdiseaseorattack", "diabetes")


# ----------------------------------------------------------
# 📊 PERFORMANCE SUMMARY
# ----------------------------------------------------------

print("\n" + "="*60)
print("📊 MODEL AUC COMPARISON")
print("="*60)

print(f"Framingham   : {auc_f:.4f}")
print(f"Cardiovascular: {auc_c:.4f}")
print(f"Heart Attack  : {auc_ha:.4f}")
print(f"Heart Failure : {auc_hf:.4f}")
print(f"Diabetes      : {auc_d:.4f}")

print("\n🚀 All models trained successfully!")


🚀 Training framingham model
Accuracy: 0.8342
ROC-AUC : 0.7093
✅ framingham model saved

🚀 Training cardio model
Accuracy: 0.7038
ROC-AUC : 0.7615
✅ cardio model saved

🚀 Training heart_attack model
Accuracy: 0.6098
ROC-AUC : 0.5001
✅ heart_attack model saved

🚀 Training heart_failure model
Accuracy: 0.7333
ROC-AUC : 0.8086
✅ heart_failure model saved

🚀 Training diabetes model
Accuracy: 0.8917
ROC-AUC : 0.7482
✅ diabetes model saved

📊 MODEL AUC COMPARISON
Framingham   : 0.7093
Cardiovascular: 0.7615
Heart Attack  : 0.5001
Heart Failure : 0.8086
Diabetes      : 0.7482

🚀 All models trained successfully!
